# Problem Set 1: Set-up and Natality Data

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

Problem Set 1 is about setting up your workflow for future assignments in this course. We are going to use **Google Colab** notebooks: your code, your results, and your written answers all live in one document — this one. There is nothing to install; you only need a browser and a Google account.

A notebook is a sequence of **cells**. Code cells contain Python code and are run by clicking the ▶ button or pressing **Shift+Enter**. Text cells (like this one) contain formatted text; double-click to edit them. When a question asks for a written answer, type it into the **Answer** cell provided below the question.

**Before you do anything else:** go to **File → Save a copy in Drive**. This gives you your own editable copy of the notebook.

**Note:** For those with experience using Python and notebooks this will be a simple assignment; for others it will be more challenging. Either way, understanding your workflow is **very important** and will make future problem sets in this course much easier.

## Part 0: Set-up

The cell below loads the packages we will use all quarter:

- `pandas` — data tables (think of it as a more powerful Excel)
- `matplotlib` — graphs
- `statsmodels` — regressions

Run it now. If it prints "Ready!", your set-up works — that's the whole installation step.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

print("Ready!")

## Part 1: Load the data

We use real Vital Statistics data on all births that occurred in the United States in 2022. Variable descriptions are available from [NBER's natality data page](https://www.nber.org/research/data/vital-statistics-natality-birth-data).

The file is in Stata format (`.dta`), which pandas can read directly from a link — no download needed.

In [ ]:
# Course data link (do not change)
DATA_URL = "https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/natl2022_small.dta"

df = pd.read_stata(DATA_URL)

# The file stores numbers in a compact format (int8) that can overflow when we
# square them later, so we convert everything to regular integers right away.
df = df.astype("int64")

print("Rows:", len(df))
df.head()

## Part 2: Clean the data

In this data, `dbwt` (birth weight in grams) uses the code `9999` for "not reported", and `meduc` (mother's education) uses `9` for "unknown". We drop those rows, give the remaining variables readable names, and create the squared-age variable we will need later.

| Code | Variable |
|---|---|
| `dob_mm` | month of birth |
| `dbwt` | birth weight (grams) |
| `mager` | mother's age |
| `meduc` | mother's education (1 = 8th grade or less, ..., 8 = doctorate) |

In [ ]:
df = df[(df["dbwt"] < 9999) & (df["meduc"] < 9)].copy()

df = df.rename(columns={
    "dob_mm": "month_of_birth",
    "dbwt":   "birth_weight",
    "mager":  "age",
    "meduc":  "edu",
})

df["age2"] = df["age"] ** 2

print("Rows after dropping missing values:", len(df))
df.describe()

## Part 3: Figures

### Figure 1: Number of births by month

First we count the number of births in each calendar month and plot them.

In [ ]:
births_by_month = df.groupby("month_of_birth").size()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(births_by_month.index, births_by_month.values, marker="o", markersize=4)
ax.set_xticks(range(1, 13))
ax.set_xlabel("Month of birth")
ax.set_ylabel("Number of births")
ax.set_title("Number of births")
plt.show()

**Question 1.** Look at the pattern in Figure 1 — what does it tell us?

**Answer:** *[double-click and type your answer here]*

### Figure 2: Birth weight by mother's education

Next we compute the *average* birth weight within each education group and plot it. The `groupby(...).mean()` step is the equivalent of collapsing the data to group means.

In [ ]:
bw_by_edu = df.groupby("edu")["birth_weight"].mean()

edu_labels = ["<=8", "9-11", "HS/GED", "13-14", "AA/AS", "BA", "MA", "PhD"]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(bw_by_edu.index, bw_by_edu.values, marker="o", markersize=4)
ax.set_xticks(range(1, 9), edu_labels)
ax.set_xlabel("Mother's education")
ax.set_ylabel("Average birth weight")
ax.set_title("Birth weight by maternal education")
plt.show()

### Figure 3: Birth weight by mother's age

In [ ]:
bw_by_age = df.groupby("age")["birth_weight"].mean()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(bw_by_age.index, bw_by_age.values, marker="o", markersize=4)
ax.set_xlabel("Mother's age")
ax.set_ylabel("Average birth weight")
ax.set_title("Birth weight by maternal age")
plt.show()

**Question 2.** What do Figures 2 and 3 tell us about the relationships of infant health with these maternal characteristics?

**Answer:** *[your answer here]*

### Figure 4: Fitted lines

Now we add a **linear** and a **quadratic** fitted line to the age profile, to see how well each functional form approximates the data. `np.polyfit(x, y, k)` finds the polynomial of degree `k` that best fits the points.

In [ ]:
x = bw_by_age.index.values
y = bw_by_age.values

linear_fit    = np.polyval(np.polyfit(x, y, 1), x)
quadratic_fit = np.polyval(np.polyfit(x, y, 2), x)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, fit, label in zip(axes, [linear_fit, quadratic_fit], ["Linear fit", "Quadratic fit"]):
    ax.plot(x, y, marker="o", markersize=4)
    ax.plot(x, fit)
    ax.set_xlabel("Mother's age")
    ax.set_title(label)
axes[0].set_ylabel("Average birth weight")
fig.suptitle("Birth weight by maternal age, with fitted lines")
plt.show()

**Question 3.** Which functional form approximates the age profile better, and why do you think that is?

**Answer:** *[your answer here]*

## Part 4: Regressions

Finally we run three regressions of birth weight on mother's characteristics, using the full data (one row per birth, not the group means):

1. birth weight on education group indicators ("dummies")
2. adding age and age² as controls
3. replacing age and age² with a full set of age dummies (a "fixed effect" for every age)

In the formulas below, `C(edu, Treatment(1))` tells statsmodels to treat education as a categorical variable with group 1 (8th grade or less) as the omitted reference category — the same thing Stata's `i.edu` does. `cov_type="HC1"` asks for robust standard errors. The last cell collects all three regressions into one comparison table.

In [ ]:
reg1 = smf.ols("birth_weight ~ C(edu, Treatment(1))", df).fit(cov_type="HC1")
reg2 = smf.ols("birth_weight ~ C(edu, Treatment(1)) + age + age2", df).fit(cov_type="HC1")
reg3 = smf.ols("birth_weight ~ C(edu, Treatment(1)) + C(age)", df).fit(cov_type="HC1")

# Full detail for the first regression (run reg2.summary() / reg3.summary() to inspect the others)
reg1.summary()

In [ ]:
keep_order = [f"C(edu, Treatment(1))[T.{i}]" for i in range(2, 9)] + ["age", "age2", "Intercept"]

table = summary_col(
    [reg1, reg2, reg3],
    stars=True,
    model_names=["(1)", "(2)", "(3)"],
    regressor_order=keep_order,
    drop_omitted=True,   # hides the 38 individual age dummies in column (3)
    info_dict={
        "Mother age dummies": lambda r: "Yes" if any("C(age)" in n for n in r.params.index) else "No",
        "N": lambda r: f"{int(r.nobs):,}",
    },
)
print(table)

**Question 4.** Looking at the results, is it surprising that the education coefficients in column (2) are smaller than in column (1)? And does it make sense that the coefficients in columns (2) and (3) are so similar?

**Answer:** *[your answer here]*

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). It sends this notebook, with all its results, directly to the teaching team. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS01"

import json, requests
from google.colab import _message

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID, "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted! Received as {result['file']} for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")